In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
compile_pgf = False
if compile_pgf:
	matplotlib.use("pgf")
	matplotlib.rcParams.update({
	    "pgf.texsystem": "pdflatex",
	    'font.family': 'serif',
	    'text.usetex': True,
	    "axes.formatter.use_mathtext": True
	})

# Experiment 1: any-start-time plans

In [ ]:
aggregate_data = True
filename_complete = Path(os.path.abspath("__file__")).parent / "complete_results_individual_delays.json"
result_file = Path(os.path.abspath("__file__")).parent / "table_flexsipp_anystartimeplans.tex"
if not aggregate_data and os.path.isfile(filename_complete):
    with open(filename_complete, "r") as f:
        all_results = json.load(f)
else:
	config = {
		"maze1": [x for x in range(0, 47)],
		"warehouse1": [x for x in range(47, 57)]
	}
	all_results = {"maze1": {"k20": {}, "k50": {}}, "warehouse1": {"k50": {}}}
	for c in config:
		data_dir = Path(os.path.abspath("__file__")).parent.parent / "db_output" / c
		for x in config[c]:
			agents = ["k20" for _ in range(25)] + ["k50" for _ in range(57-25)]
			if x in [32, 36, 42]:
				print(f"Scenarios for {c} {agents[x]} do not exist with idx {x}")
				continue
			scen_name = f"{c}-{agents[x]}-{x}"
			all_results[c][agents[x]][scen_name] = {"FlexSIPP": {"oFalse": {"0": [], "3": [], "5": [], "8": []}, "oTrue": {"0": [], "3": [], "5": [], "8": []}}, "@MAEDeR": {"oFalse": {"0": [], "3": [], "5": [], "8": []}, "oTrue": {"0": [], "3": [], "5": [], "8": []}}}
			for a in ["FlexSIPP", "@MAEDeR"]:
				pattern = f"single_{c}_{a}_2026-07-29-*_{x}-56.json"
				scenario_file = data_dir.rglob(pattern)
				file = next(scenario_file, None)
				if file:
					with open(file, "r") as f:
						data = json.load(f)
						for key, data_value in data.items():
							info = key.split("_")
							optimize = info[-1]
							num_run = info[-2]
							if info[-3] == "paths":
								flex = 0
							else:
								flex = info[-3]
							all_results[c][agents[x]][scen_name][a][f"o{optimize}"][f"{flex}"].append(data_value["delay0"])
				else:		
					print("Could not find", pattern)
	with open(filename_complete, "w") as f:
		json.dump(all_results, f, indent=4)

In [ ]:
df = pd.DataFrame(columns=["Map", "Scen", "Agents", "Flex", "Optimal", "Iter", "Delay", "FlexSIPP", "@MAEDeR", "Delay Diff", "Delay Comp", "Time F", "Time M"])
not_found = {"@MAEDeR": 0, "FlexSIPP": 0}
found = {"@MAEDeR": 0, "FlexSIPP": 0}
num_rows = 0
for map_name, num_agents in [("maze1", "k20"), ("maze1", "k50"), ("warehouse1", "k50")]:
	for flex in ["0", "3", "5", "8"]:
		for opt in ["oTrue", "oFalse"]:
			for scenario in all_results[map_name][num_agents]:
				for idx in range(3):
					flex_row = all_results[map_name][num_agents][scenario]["FlexSIPP"][opt][flex][idx]
					maeder_row = all_results[map_name][num_agents][scenario]["@MAEDeR"][opt][flex][idx]
					if "unique_routes_safe" in maeder_row and maeder_row["unique_routes_safe"]:
						total_delay_maeder = sum([maeder_row["final_paths"][a]["arrival"][1] - maeder_row["initial_paths"][a]["arrival"][1] for a in maeder_row["final_paths"]])
						found["@MAEDeR"] += 1
					else:
						total_delay_maeder = np.nan
						not_found["@MAEDeR"] += 1
					if "unique_routes_safe" in flex_row and flex_row["unique_routes_safe"]:
						total_delay_flexsipp = sum([flex_row["final_paths"][a]["arrival"][1] - flex_row["initial_paths"][a]["arrival"][1] for a in flex_row["final_paths"]])
						found["FlexSIPP"] += 1
					else:
						total_delay_flexsipp = np.nan
						not_found["FlexSIPP"] += 1
					diff = np.nan
					comp = np.nan
					if not np.isnan(total_delay_maeder) and not np.isnan(total_delay_flexsipp):
						diff = total_delay_flexsipp - total_delay_maeder
						if total_delay_maeder == 0:
							if total_delay_flexsipp == 0:
								comp = 100
							else:
								comp = 0
						else: 
							comp = total_delay_flexsipp / total_delay_maeder * 100
					df.loc[num_rows] = [
						map_name,
						scenario,
						num_agents,
						flex,
						opt, 
						int(idx),
						flex_row["delay"], # the input delay
						total_delay_flexsipp,
						total_delay_maeder,
						diff,
						comp,
						flex_row["Search Time"] / 1000,
						maeder_row["Search Time"] / 1000
					]
					num_rows += 1
print("not found", not_found)
print("found", found)
df

In [ ]:
df["Better"] = (df["@MAEDeR"] < df["FlexSIPP"]) > 0
df["Scenarios"] = df.apply(lambda x: f'{x["Agents"][1:]}', axis=1)
df = df[df["Optimal"] == "oTrue"]
scenario_groups = df.groupby(["Map", "Scenarios", "Flex"])
times = scenario_groups[["Delay Comp", "Time F", "Time M"]].mean(skipna=True)
times = times[["Delay Comp",  'Time F', 'Time M']].round(2)
percentage_found = (scenario_groups[["FlexSIPP", "@MAEDeR"]]
                    .apply(lambda x: x.count() / (x.count() + x.isna().sum()) * 100)
                    .astype(int).astype(str) + r'\%')
times = times.join(percentage_found)
times["Delay Comp"] = times["Delay Comp"].apply(lambda x: str(round(x, 1)) + r"\%")

with open(result_file, "w") as f:
    latex = times.to_latex(
        float_format="%.2f",
        columns=["Delay Comp", "Time F", "Time M", "FlexSIPP", "@MAEDeR"],
        column_format="ccclcccc",
        multirow=True,
        multicolumn=True,
        caption=r'Total delay comparison for maze scenarios with 20 and 50 ($\numAgents$) agents and warehouse (ware) scenarios with 50 agents, where one agent is delayed and extra flexibility (\IntroducedFlexibility{}) is added in the initial paths. Shows the average delay difference in percentage (FlexSIPP / @MAEDeR) and the percentage of runs when a path was found, along with the average search time over those for FlexSIPP (F) and @MAEDeR (M).', 
        label="tab:delays", 
        position="t"
    )
    lines = latex.split("\n")
    for i, line in enumerate(lines):
        lines[i] = line.replace("maze1", "maze")
        lines[i] = lines[i].replace(r"\multirow[t]{4}{*}{warehouse1}", r"\multirow[c]{4}{*}{\shortstack[l]{ware\\house}}")
        if "Delay Comp" in line:
            lines[i] = line.replace("Delay Comp", r'\textbf{Diff}').replace(r'Time F & Time M', r'\multicolumn{2}{c}{\textbf{Time (s)}}').replace(r'FlexSIPP & @MAEDeR', r'\multicolumn{2}{c}{\textbf{Path Found}}')
            lines[i+1] = r' & '.join(lines[i+1].replace(r'Scenarios', r'$\numAgents$').replace(r'Flex', r'\IntroducedFlexibility{}').split(r' & ')[:4]) + r' & ' + r'F & M & F & M \\'
        if r'\cline' in line:
            if r'\bottomrule' in lines[i+1]:
                lines.pop(i)
            else:
                lines[i] = line.replace(r'\cline', r'\cmidrule')
    print('\n'.join(lines))
    f.write('\n'.join(lines))
times

# Experiment 2: Sequential delays

In [ ]:
aggregate_sequential_data = True
seq_filename_complete = Path(os.path.abspath("__file__")).parent / "complete_results_sequential_delays.json"

if not aggregate_data and os.path.isfile(seq_filename_complete):
    with open(seq_filename_complete, "r") as f:
    	complete_result = json.load(f)
else:
    filename = "replan_@MAEDeR_maze-128-128-1-even-1-k50_2026-07-30-09-28_f0_oTrue_seed42_25delays_mINF"
    folder = "db_output"
    if "FlexSIPP" in filename:
        raise ValueError("Should enter the @MAEDeR file not the FlexSIPP file")
    filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", folder, f"{filename}.json")
    complete_result = {
		"@MAEDeR": json.load(open(filepath, "r")),
		"FlexSIPP": json.load(open(filepath.replace("@MAEDeR", "FlexSIPP"), "r"))
	}
    with open(seq_filename_complete, "w") as f:
        json.dump(complete_result, f, indent=4)

In [ ]:
df_seq = pd.DataFrame(columns=["Delay Idx", "Delay Agent", "Delayed Starttime", "Delay Amount", "Cumulative Delay", "Total FlexSIPP" ,"Total @MAEDeR", "Fail FlexSIPP", "Fail @MAEDeR"])
rows = 0
paths = {
    "FlexSIPP": {a: [r["arrival"][1]]  for a, r in complete_result["@MAEDeR"]["delay0"]["initial_paths"].items()},
    "@MAEDeR": {a: [r["arrival"][1]]  for a, r in complete_result["@MAEDeR"]["delay0"]["initial_paths"].items()}
}
delays = ["" for a in complete_result["@MAEDeR"]["delay0"]["initial_paths"]]
cumulative_delay = 0
for delay in complete_result["FlexSIPP"]:
    if complete_result["FlexSIPP"][delay]:
        for a, r in complete_result["FlexSIPP"][delay]["arrival_times"].items():
            paths["FlexSIPP"][a].append(r["arrival"][1])
        for a, r in complete_result["@MAEDeR"][delay]["arrival_times"].items():
            paths["@MAEDeR"][a].append(r["arrival"][1])
        delays[int(complete_result["FlexSIPP"][delay]["delay_agent"])-1] = str(complete_result["FlexSIPP"][delay]["delay_agent"])
        assert complete_result["FlexSIPP"][delay]["delay_agent"] == complete_result["@MAEDeR"][delay]["delay_agent"]
        cumulative_delay += (complete_result["FlexSIPP"][delay]["delayed_start_time"] - complete_result["FlexSIPP"][delay]["original_start_time"])
        print(f"{rows}: Delay {delay} has start time {complete_result['FlexSIPP'][delay]['original_start_time']} and delayed start {complete_result['FlexSIPP'][delay]['delayed_start_time']}, cumulative delay={cumulative_delay}")
        df_seq.loc[rows] = [
			rows,
			complete_result["FlexSIPP"][delay]["delay_agent"],
			complete_result["FlexSIPP"][delay]["delayed_start_time"],
			complete_result["FlexSIPP"][delay]["delayed_start_time"] - complete_result["FlexSIPP"][delay]["original_start_time"],
			cumulative_delay,
			sum([(paths["FlexSIPP"][a][-1] - paths["FlexSIPP"][a][0]) for a in paths["FlexSIPP"]]),
			sum([(paths["@MAEDeR"][a][-1] - paths["@MAEDeR"][a][0]) for a in paths["@MAEDeR"]]),
			not (complete_result["FlexSIPP"][delay] and complete_result["FlexSIPP"][delay]["unique_routes_safe"]),
			not (complete_result["@MAEDeR"][delay] and complete_result["@MAEDeR"][delay]["unique_routes_safe"]),
		]
        rows += 1

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

colors = [
    (0.83527, 0.886029, 0.102646),
    (0.283187, 0.125848, 0.44496),
    (0.132268, 0.655014, 0.519661),
]
print("Max @MAEDeR", df_seq['Total @MAEDeR'].max(), "Max FlexSIPP", df_seq['Total FlexSIPP'].max(), "FlexSIPP / @MAEDeR = ", round(df_seq['Total FlexSIPP'].max() / df_seq['Total @MAEDeR'].max() * 100, 2), "%")

df_seq.plot(ax=ax, x="Delay Idx", y="Cumulative Delay", label="Input Delay", color=colors[0])
df_seq.plot(ax=ax, x="Delay Idx", y="Total FlexSIPP", label="Total Delay FlexSIPP", color=colors[1])
df_seq.plot(ax=ax, x="Delay Idx", y="Total @MAEDeR", label="Total Delay @MAEDeR", color=colors[2])


ax.scatter(df_seq.index[df_seq['Fail FlexSIPP']], df_seq.loc[df_seq['Fail FlexSIPP'], 'Total FlexSIPP'], marker='x', color=colors[1], zorder=5, linewidth=2)
ax.scatter(df_seq.index[df_seq['Fail @MAEDeR']], df_seq.loc[df_seq['Fail @MAEDeR'], 'Total @MAEDeR'], marker='x', color=colors[2], zorder=5, linewidth=2)

ticks = [int(x) + 1 if i % 2 == 0 else "" for i, x in enumerate(df_seq["Delay Idx"].unique())]

fonts = 16

ax.set_xticks(range(len(df_seq["Delay Idx"].unique())))
ax.set_xticklabels(ticks, fontsize=fonts)
ax.set_ylabel("Total Delay", fontsize=fonts)
ax.set_xlabel("Delay Index", fontsize=fonts)
ax.legend(fontsize=fonts)
ax.set_xlabel(ax.get_xlabel(), fontsize=fonts)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=fonts)
filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "sequential_delay_updates")
plt.tight_layout()
extension = "pgf" if compile_pgf else "png"
plt.savefig(f"{filepath}.{extension}", dpi=600)
plt.show()
plt.close()
print("Max @MAEDeR", df_seq['Total @MAEDeR'].max(), "Max FlexSIPP", df_seq['Total FlexSIPP'].max(), "FlexSIPP / @MAEDeR = ", round(df_seq['Total FlexSIPP'].max() / df_seq['Total @MAEDeR'].max() * 100, 2), "%")